# 🎯 Sprint 3 — Hyperparameter Tuning & Final Model

**Objective:** Tune the top models from Sprint 2/3, select the best, build the full sklearn Pipeline, and serialise for deployment.

---

In [11]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data.load_data import load_raw_data
from src.data.clean_data import clean_data
from src.features.engineer import engineer_features
from src.models.tune import random_search_tune
from src.models.evaluate import (
    evaluate_classifier, plot_confusion_matrix, plot_roc_curves,
    classification_report_df
)
from src.pipeline.build_pipeline import build_pipeline, save_pipeline
from src.utils.config import (
    TARGET, RANDOM_SEED, TEST_SIZE, FIGURES_DIR,
    NUMERIC_FEATURES, CATEGORICAL_LOW_CARD, CATEGORICAL_HIGH_CARD
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Prepare Data (with Engineered Features)

In [12]:
df = clean_data(load_raw_data())
df = engineer_features(df)

# Drop lead_time_category (handled by pipeline as ordinal)
if 'lead_time_category' in df.columns:
    df = df.drop(columns=['lead_time_category'])

y = df[TARGET]
X = df.drop(columns=[TARGET])

# Update feature lists with engineered numeric features
engineered_numeric = [
    'total_stay', 'total_guests', 'is_local', 'adr_per_person',
    'is_same_room', 'booking_changes_flag', 'is_family',
    'arrival_month_sin', 'arrival_month_cos',
    'total_previous', 'cancellation_ratio'
]
all_numeric = NUMERIC_FEATURES + [c for c in engineered_numeric if c in X.columns]

# Train-test split (we'll tune on train, evaluate on held-out test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

2026-08-25 15:21:30 │ INFO     │ src.data.load_data │ Loading raw data from C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\hotel_bookings.csv


2026-08-25 15:21:31 │ INFO     │ src.data.load_data │ Shape: (119390, 32)
2026-08-25 15:21:31 │ INFO     │ src.data.load_data │ Columns: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']
2026-08-25 15:21:31 │ WARNING  │ src.data.load_data │ Columns with nulls:
children         4
country        488
agent        16340
company     112593
2026-08-25 15:21:31 │ INFO     │ src.data.load_data │ Duplicate rows: 31994
2026-08-25 15:21:3

c:\Users\pares\OneDrive\Desktop\hotle_cnacelation\src\data\clean_data.py:68: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["children"].fillna(0, inplace=True)
c:\Users\pares\OneDrive\Desktop\hotle_cnacelation\src\data\clean_data.py:71: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy

2026-08-25 15:21:34 │ INFO     │ src.data.clean_data │ Removed 181 invalid rows (negative ADR / zero guests)
2026-08-25 15:21:34 │ INFO     │ src.data.clean_data │ Converted arrival_date_month to numeric (1–12)
2026-08-25 15:21:34 │ INFO     │ src.data.clean_data │ Dropped 32246 duplicate rows
2026-08-25 15:21:35 │ INFO     │ src.data.clean_data │ Cleaned data saved to C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\data\interim\hotel_bookings_cleaned.csv  (output shape: (86963, 29))
2026-08-25 15:21:35 │ INFO     │ src.features.engineer │ Engineered 12 new features → 41 total columns
Train: (69570, 39), Test: (17393, 39)


## 2. Build Pipeline & Tune Random Forest

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = build_pipeline(
    RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    numeric_features=all_numeric,
    categorical_low_card=CATEGORICAL_LOW_CARD,
    categorical_high_card=CATEGORICAL_HIGH_CARD,
)

rf_params = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [10, 20, 30, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
}

rf_search = random_search_tune(
    rf_pipeline, rf_params, X_train, y_train,
    n_iter=30, scoring='f1', use_mlflow=False
)
print(f'\n🏆 Best RF params: {rf_search.best_params_}')
print(f'   Best F1: {rf_search.best_score_:.4f}')

2026-08-25 15:21:36 │ INFO     │ src.pipeline.build_pipeline │ Built preprocessor: 29 numeric, 8 cat_low, 1 cat_high features
2026-08-25 15:21:36 │ INFO     │ src.pipeline.build_pipeline │ Built pipeline with classifier: RandomForestClassifier
2026-08-25 15:21:36 │ INFO     │ src.models.tune │ RandomizedSearchCV on Pipeline — 30 iterations × 5 folds
Fitting 5 folds for each of 30 candidates, totalling 150 fits
2026-08-25 16:01:10 │ INFO     │ src.models.tune │ Best params: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 30}
2026-08-25 16:01:10 │ INFO     │ src.models.tune │ Best f1: 0.7041

🏆 Best RF params: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 30}
   Best F1: 0.7041


## 3. Tune XGBoost (if available)

In [ ]:
try:
    from xgboost import XGBClassifier
    
    xgb_pipeline = build_pipeline(
        XGBClassifier(
            use_label_encoder=False, eval_metric='logloss',
            random_state=RANDOM_SEED, n_jobs=-1
        ),
        numeric_features=all_numeric,
        categorical_low_card=CATEGORICAL_LOW_CARD,
        categorical_high_card=CATEGORICAL_HIGH_CARD,
    )
    
    xgb_params = {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [3, 5, 7, 10],
        'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'classifier__subsample': [0.7, 0.8, 0.9, 1.0],
        'classifier__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    }
    
    xgb_search = random_search_tune(
        xgb_pipeline, xgb_params, X_train, y_train,
        n_iter=30, scoring='f1', use_mlflow=False
    )
    print(f'\n🏆 Best XGB params: {xgb_search.best_params_}')
    print(f'   Best F1: {xgb_search.best_score_:.4f}')
    
    HAS_XGB = True
except ImportError:
    print('⚠️ XGBoost not installed — skipping')
    HAS_XGB = False

2026-08-25 16:01:10 │ INFO     │ src.pipeline.build_pipeline │ Built preprocessor: 29 numeric, 8 cat_low, 1 cat_high features
2026-08-25 16:01:10 │ INFO     │ src.pipeline.build_pipeline │ Built pipeline with classifier: XGBClassifier


2026-08-25 16:01:10 │ INFO     │ src.models.tune │ RandomizedSearchCV on Pipeline — 30 iterations × 5 folds
Fitting 5 folds for each of 30 candidates, totalling 150 fits


## 4. Tune LightGBM (if available)

In [ ]:
try:
    from lightgbm import LGBMClassifier
    
    lgbm_pipeline = build_pipeline(
        LGBMClassifier(random_state=RANDOM_SEED, n_jobs=-1, verbose=-1),
        numeric_features=all_numeric,
        categorical_low_card=CATEGORICAL_LOW_CARD,
        categorical_high_card=CATEGORICAL_HIGH_CARD,
    )
    
    lgbm_params = {
        'classifier__n_estimators': [100, 200, 300, 500],
        'classifier__max_depth': [3, 5, 7, 10, -1],
        'classifier__learning_rate': [0.01, 0.05, 0.1],
        'classifier__num_leaves': [31, 50, 70, 100],
        'classifier__subsample': [0.7, 0.8, 0.9, 1.0],
    }
    
    lgbm_search = random_search_tune(
        lgbm_pipeline, lgbm_params, X_train, y_train,
        n_iter=30, scoring='f1', use_mlflow=False
    )
    print(f'\n🏆 Best LGBM params: {lgbm_search.best_params_}')
    print(f'   Best F1: {lgbm_search.best_score_:.4f}')
    
    HAS_LGBM = True
except ImportError:
    print('⚠️ LightGBM not installed — skipping')
    HAS_LGBM = False

⚠️ LightGBM not installed — skipping


## 5. Select Best Model & Final Evaluation

In [ ]:
# Collect all tuned models
candidates = {'RandomForest': rf_search}
if HAS_XGB:
    candidates['XGBoost'] = xgb_search
if HAS_LGBM:
    candidates['LightGBM'] = lgbm_search

# Evaluate on held-out test set
final_results = {}
for name, searcher in candidates.items():
    best_pipeline = searcher.best_estimator_
    metrics = evaluate_classifier(best_pipeline, X_test, y_test)
    final_results[name] = metrics
    print(f'{name}: {metrics}')

final_df = pd.DataFrame(final_results).T.sort_values('f1', ascending=False)
final_df.style.background_gradient(cmap='Greens')

RandomForest: {'accuracy': 0.8506, 'precision': 0.7695, 'recall': 0.6465, 'f1': 0.7027, 'roc_auc': 0.9115}
XGBoost: {'accuracy': 0.857, 'precision': 0.7604, 'recall': 0.6956, 'f1': 0.7266, 'roc_auc': 0.922}


,accuracy,precision,recall,f1,roc_auc
XGBoost,0.857000,0.760400,0.695600,0.726600,0.922000
RandomForest,0.850600,0.769500,0.646500,0.702700,0.911500


In [ ]:
# Pick the best
best_name = final_df.index[0]
best_pipeline = candidates[best_name].best_estimator_
best_metrics = final_results[best_name]

print(f'\n🏆 BEST MODEL: {best_name}')
print(f'   Metrics: {best_metrics}')

# Confusion matrix
plot_confusion_matrix(
    best_pipeline, X_test, y_test,
    model_name=f'{best_name}_tuned',
    save_path=FIGURES_DIR / f'confusion_matrix_{best_name}_tuned.png'
)

# Classification report
report = classification_report_df(best_pipeline, X_test, y_test)
report


🏆 BEST MODEL: XGBoost
   Metrics: {'accuracy': 0.857, 'precision': 0.7604, 'recall': 0.6956, 'f1': 0.7266, 'roc_auc': 0.922}
2026-08-25 01:40:30 │ INFO     │ src.models.evaluate │ Confusion matrix saved to C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\reports\figures\confusion_matrix_XGBoost_tuned.png


,precision,recall,f1-score,support
0,0.889178,0.917662,0.903196,12643.000000
1,0.760414,0.695579,0.726553,4750.000000
accuracy,0.857011,0.857011,0.857011,0.857011
macro avg,0.824796,0.806620,0.814874,17393.000000
weighted avg,0.854013,0.857011,0.854955,17393.000000


## 6. Save Final Pipeline

In [ ]:
# Re-fit best pipeline on ALL training data
best_pipeline.fit(X_train, y_train)

save_pipeline(best_pipeline, best_metrics)
print(f'\n✅ Final pipeline ({best_name}) saved to models/')
print(f'   Model: models/model.pkl')
print(f'   Metadata: models/model_metadata.json')

c:\Users\pares\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [01:40:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-08-25 01:40:39 │ INFO     │ src.pipeline.build_pipeline │ Pipeline saved to C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\models\model.pkl
2026-08-25 01:40:39 │ INFO     │ src.pipeline.build_pipeline │ Metadata saved to C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\models\model_metadata.json

✅ Final pipeline (XGBoost) saved to models/
   Model: models/model.pkl
   Metadata: models/model_metadata.json


## Summary

- Tuned Random Forest, XGBoost, and LightGBM with RandomizedSearchCV
- Selected the best model based on F1 score on held-out test set
- Saved the full sklearn Pipeline (preprocessor + classifier) for deployment
- Model metadata (metrics, features, seed) saved as JSON

In [ ]:
pip install streamlit


  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
